# 实验一（akshare 版）：A股银行股数据获取

## 依赖
```
pip install akshare pandas
```

## 数据源
全部数据通过 **akshare**（东方财富公开接口聚合）获取：

| 案例 | API | 说明 |
|------|-----|------|
| 1. 银行板块清单 | `ak.stock_board_industry_cons_em(symbol="银行")` | 东方财富行业板块成分股 |
| 2. 日线行情 | `ak.stock_zh_a_hist(symbol, period, start_date, end_date, adjust)` | 支持前复权/后复权/不复权 |
| 3. 分红配送 | `ak.stock_fhps_detail_em(symbol)` | 个股历史分红配送详情 |
| 4. 复权对比 | `ak.stock_zh_a_hist(adjust="")` vs `adjust="qfq"` | 不复权 vs 前复权 |

## 目标股票
四大行（工商、建设、农业、中国）+ 招商银行，2022 年 1 月至今

## 复权说明
- **前复权 (qfq)**：历史价格按除权比例下调，最新价不变 → 适合回测
- **后复权 (hfq)**：历史价格按除权比例上调，最早价不变 → 适合看真实涨幅
- **不复权**：原始交易价格，含除权缺口

---
## 0. 环境准备

In [1]:
import os
import time
from contextlib import contextmanager

import akshare as ak
import pandas as pd


# 本 Notebook 当前使用的 AKShare 接口均来自东方财富。
# 只绕过这些域名；其他请求继续遵循用户原有的系统代理配置。
AKSHARE_DIRECT_DOMAINS = ("eastmoney.com",)


@contextmanager
def akshare_no_proxy():
    """仅让当前 AKShare 数据源直连，并精确恢复原 NO_PROXY。"""
    no_proxy_keys = ("NO_PROXY", "no_proxy")
    original_env = {
        key: os.environ.get(key)
        for key in no_proxy_keys
    }
    existing = [
        value.strip(" ,")
        for value in original_env.values()
        if value and value.strip(" ,")
    ]
    bypass_value = ",".join(
        dict.fromkeys([*existing, *AKSHARE_DIRECT_DOMAINS])
    )

    try:
        for key in no_proxy_keys:
            os.environ[key] = bypass_value
        yield
    finally:
        for key in no_proxy_keys:
            os.environ.pop(key, None)
        for key, value in original_env.items():
            if value is not None:
                os.environ[key] = value


# 无论从仓库根目录还是 labs/ 启动 Jupyter，都固定写入 labs/data。
WORKING_DIR = os.path.abspath("")
LABS_DIR = (
    WORKING_DIR
    if os.path.basename(WORKING_DIR) == "labs"
    else os.path.join(WORKING_DIR, "labs")
)
if not os.path.isfile(os.path.join(LABS_DIR, "pyproject.toml")):
    raise FileNotFoundError("请从仓库根目录或 labs/ 启动 Jupyter")
DATA_DIR = os.path.join(
    LABS_DIR,
    "data",
    "lab1_akshare",
)
os.makedirs(DATA_DIR, exist_ok=True)

# 目标股票
TARGET_STOCKS = {
    "601398": "工商银行",
    "601939": "建设银行",
    "601288": "农业银行",
    "601988": "中国银行",
    "600036": "招商银行",
}

START_DATE = "20220101"
END_DATE = "20260720"

print(f"AKShare 版本: {ak.__version__}")
print(f"数据目录: {DATA_DIR}")
print(f"目标股票: {len(TARGET_STOCKS)} 只")

AKShare 版本: 1.18.74
数据目录: D:\vibe-coding\trading-topic\labs\data\lab1_akshare
目标股票: 5 只


---
## 工具函数

In [2]:
def safe_call(fn, label, *args, retries=3, delay=2.0, **kwargs):
    """调用 AKShare；失败时重试，全部失败返回 None。"""
    last_err = None

    for attempt in range(1, retries + 1):
        try:
            with akshare_no_proxy():
                result = fn(*args, **kwargs)

            print(f"  [√] {label}")
            return result

        except Exception as exc:
            last_err = exc

            if attempt < retries:
                wait = delay * attempt
                print(
                    f"  [?] {label} 重试 "
                    f"{attempt}/{retries}，等待 {wait:.0f} 秒"
                )
                time.sleep(wait)

    print(
        f"  [!] {label} 失败: "
        f"{type(last_err).__name__}: {last_err}"
    )
    return None

---
## 案例 1：获取银行板块股票清单

**API**: `ak.stock_board_industry_cons_em(symbol="银行")`

- `symbol` 支持板块名称（如 `"银行"`）或板块代码（如 `"BK1027"`）
- 可用 `ak.stock_board_industry_name_em()` 查看全部行业板块名称
- 返回列：序号 代码 名称 最新价 涨跌幅 涨跌额 成交量 成交额 振幅 最高 最低 今开 昨收 换手率 市盈率-动态 市净率（**不含总市值**）

> **注意**：如果遇到 `ConnectionError`，通常是东方财富服务器临时限流，等待几分钟后重试即可。

In [5]:
stock_board_industry_name_em_df = safe_call(
    ak.stock_board_industry_name_em,
    "行业板块名称",
    retries=2,
    delay=3,
)

if stock_board_industry_name_em_df is not None:
    display(stock_board_industry_name_em_df.head())

AttributeError: module 'akshare' has no attribute 'stock_board_industry_name'

In [4]:
print("=" * 65)
print("案例 1：获取 A 股银行板块股票清单")
print("=" * 65)

bank_df = safe_call(
    ak.stock_board_industry_cons_em,
    "银行板块成分股",
    symbol="银行",
    retries=2,
    delay=3,
)

if bank_df is not None and not bank_df.empty:
    print(f"\n银行板块共 {len(bank_df)} 只股票")
    display(bank_df.head())

    csv_path = os.path.join(DATA_DIR, "银行板块清单.csv")
    bank_df.to_csv(
        csv_path,
        index=False,
        encoding="utf-8-sig",
    )

    print(f"\n已保存：{csv_path}")
else:
    print("银行板块数据获取失败")

案例 1：获取 A 股银行板块股票清单
  [?] 银行板块成分股 重试 1/2，等待 3 秒
  [!] 银行板块成分股 失败: ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
银行板块数据获取失败


---
## 案例 2：四大行 + 招商银行日线行情（前复权）

**API**: `ak.stock_zh_a_hist(symbol, period, start_date, end_date, adjust)`

| 参数 | 说明 | 可选值 |
|------|------|--------|
| `symbol` | 股票代码 | 6 位字符串 |
| `period` | K 线周期 | `"daily"` `"weekly"` `"monthly"` |
| `start_date` | 起始日期 | `"YYYYMMDD"` |
| `end_date` | 结束日期 | `"YYYYMMDD"` |
| `adjust` | 复权方式 | `"qfq"`(前复权) `"hfq"`(后复权) `""`(不复权) |

In [5]:
print("\n" + "=" * 65)
print("  案例 2：四大行 + 招商银行日线行情（前复权）")
print(f"  时间范围：{START_DATE[:4]}-{START_DATE[4:6]} → 最新")
print("=" * 65)

hist_data = {}  # code → DataFrame

for code, name in TARGET_STOCKS.items():
    print(f"\n  → {name}({code}) 日线行情（前复权）...")

    df = safe_call(
        ak.stock_zh_a_hist,
        f"{name} 日线",
        symbol=code,
        period="daily",
        start_date=START_DATE,
        end_date=END_DATE,
        adjust="qfq",
        retries=3,
        delay=2,
    )

    if df is not None and not df.empty:
        # 列名整理：日期 开盘 收盘 最高 最低 成交量 成交额 振幅 涨跌幅 涨跌额 换手率
        df = df.rename(columns={
            "日期": "日期", "开盘": "开盘", "收盘": "收盘",
            "最高": "最高", "最低": "最低", "成交量": "成交量",
            "成交额": "成交额", "振幅": "振幅(%)",
            "涨跌幅": "涨跌幅(%)", "涨跌额": "涨跌额", "换手率": "换手率(%)",
        })
        df["日期"] = pd.to_datetime(df["日期"])
        df = df.sort_values("日期").reset_index(drop=True)

        csv_path = os.path.join(DATA_DIR, f"{name}_{code}_日线_前复权.csv")
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")
        print(f"    → {len(df)} 行 → labs/data/lab1_akshare/{name}_{code}_日线_前复权.csv")
        print(f"    首日: {df['日期'].iloc[0].strftime('%Y-%m-%d')}  "
              f"开盘 {df['开盘'].iloc[0]:.2f}  收盘 {df['收盘'].iloc[0]:.2f}")
        print(f"    最新: {df['日期'].iloc[-1].strftime('%Y-%m-%d')}  "
              f"开盘 {df['开盘'].iloc[-1]:.2f}  收盘 {df['收盘'].iloc[-1]:.2f}")

        hist_data[code] = df
    else:
        print(f"    [!] {name} 数据获取失败")

print(f"\n  → 共获取 {len(hist_data)} / {len(TARGET_STOCKS)} 只股票日线数据")


  案例 2：四大行 + 招商银行日线行情（前复权）
  时间范围：2022-01 → 最新

  → 工商银行(601398) 日线行情（前复权）...
  [√] 工商银行 日线
    → 1099 行 → labs/data/lab1_akshare/工商银行_601398_日线_前复权.csv
    首日: 2022-01-04  开盘 3.12  收盘 3.15
    最新: 2026-07-20  开盘 7.50  收盘 7.74

  → 建设银行(601939) 日线行情（前复权）...
  [√] 建设银行 日线
    → 1099 行 → labs/data/lab1_akshare/建设银行_601939_日线_前复权.csv
    首日: 2022-01-04  开盘 3.91  收盘 3.97
    最新: 2026-07-20  开盘 9.98  收盘 10.44

  → 农业银行(601288) 日线行情（前复权）...
  [√] 农业银行 日线
    → 1099 行 → labs/data/lab1_akshare/农业银行_601288_日线_前复权.csv
    首日: 2022-01-04  开盘 1.79  收盘 1.79
    最新: 2026-07-20  开盘 6.35  收盘 6.61

  → 中国银行(601988) 日线行情（前复权）...
  [√] 中国银行 日线
    → 1099 行 → labs/data/lab1_akshare/中国银行_601988_日线_前复权.csv
    首日: 2022-01-04  开盘 1.90  收盘 1.91
    最新: 2026-07-20  开盘 5.92  收盘 6.08

  → 招商银行(600036) 日线行情（前复权）...
  [√] 招商银行 日线
    → 1099 行 → labs/data/lab1_akshare/招商银行_600036_日线_前复权.csv
    首日: 2022-01-04  开盘 39.46  收盘 39.10
    最新: 2026-07-20  开盘 37.95  收盘 38.91

  → 共获取 5 / 5 只股票日线数据


---
## 案例 3：个股分红配送详情

**API**: `ak.stock_fhps_detail_em(symbol)` — 单只股票历史分红配送详情

> ⚠️ 注意区分：`stock_fhps_em(date="2024")` 是按年份查**全市场**数据；
> 个股查询用 `stock_fhps_detail_em(symbol="601398")`

关键返回字段：
- `现金分红-现金分红比例`：如 2.5 表示「10 派 2.5 元」，即每股 0.25 元
- `现金分红-股息率`：百分比，如 3.5 表示 3.5%
- `方案进度`：预案 / 股东大会通过 / **实施**（只有实施才是已到账）

In [6]:
print("\n" + "=" * 65)
print("  案例 3：个股分红配送详情")
print("=" * 65)

all_dividends = {}  # code → DataFrame

for code, name in TARGET_STOCKS.items():
    print(f"\n  → {name}({code}) 分红记录...")

    div_df = safe_call(
        ak.stock_fhps_detail_em,
        f"{name} 分红",
        symbol=code,
        retries=3,
        delay=2,
    )

    if div_df is not None and not div_df.empty:
        div_df.columns = [str(c).strip() for c in div_df.columns]

        # 重命名：接口返回 19 列，这里映射关键列
        col_rename = {
            "报告期": "报告期",
            "业绩披露日期": "业绩披露日期",
            "送转股份-送转总比例": "送转总比例",
            "送转股份-送股比例": "送股比例",
            "送转股份-转股比例": "转股比例",
            "现金分红-现金分红比例": "现金分红比例",
            "现金分红-现金分红比例描述": "分红描述",
            "现金分红-股息率": "股息率(%)",
            "每股收益": "每股收益",
            "每股净资产": "每股净资产",
            "每股公积金": "每股公积金",
            "每股未分配利润": "每股未分配利润",
            "净利润同比增长": "净利润同比(%)",
            "总股本": "总股本",
            "预案公告日": "预案公告日",
            "股权登记日": "股权登记日",
            "除权除息日": "除权除息日",
            "方案进度": "方案进度",
            "最新公告日期": "最新公告日期",
        }
        div_df = div_df.rename(columns={k: v for k, v in col_rename.items() if k in div_df.columns})

        # 只保留 2022 年后且已实施的方案
        if "除权除息日" in div_df.columns:
            div_df["除权除息日"] = pd.to_datetime(div_df["除权除息日"], errors="coerce")
            div_df = div_df[div_df["除权除息日"] >= "2022-01-01"].copy()
            div_df = div_df.sort_values("除权除息日", ascending=False).reset_index(drop=True)

        if "方案进度" in div_df.columns:
            div_df = div_df[div_df["方案进度"].str.contains("实施", na=False)]

        # 保存
        csv_path = os.path.join(DATA_DIR, f"{name}_{code}_分红记录.csv")
        div_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
        print(f"    → {len(div_df)} 条已实施 → labs/data/lab1_akshare/{name}_{code}_分红记录.csv")

        # 摘要
        if "现金分红比例" in div_df.columns:
            total = pd.to_numeric(div_df["现金分红比例"], errors="coerce").sum()
            print(f"    2022 年以来累计: 10 派 {total:.2f} 元（含税），每股 {total/10:.2f} 元")
        if "股息率(%)" in div_df.columns:
            avg_yield = pd.to_numeric(div_df["股息率(%)"], errors="coerce").mean()
            print(f"    平均股息率: {avg_yield:.2f}%")

        show_cols = [c for c in ["除权除息日", "现金分红比例", "股息率(%)", "送股比例", "转股比例"]
                     if c in div_df.columns]
        if show_cols:
            display(div_df[show_cols].head(5))

        all_dividends[code] = div_df
    else:
        print(f"    [!] {name} 分红数据获取失败")

print(f"\n  → 共获取 {len(all_dividends)} / {len(TARGET_STOCKS)} 只股票分红数据")


  案例 3：个股分红配送详情

  → 工商银行(601398) 分红记录...
  [?] 工商银行 分红 重试 1/3，等待 2 秒


Exception ignored in: <function tqdm.__del__ at 0x000002BCC5A8EE80>
Traceback (most recent call last):
  File "D:\vibe-coding\trading-topic\labs\.venv\Lib\site-packages\tqdm\std.py", line 1154, in __del__
    self.close()
  File "D:\vibe-coding\trading-topic\labs\.venv\Lib\site-packages\tqdm\notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
    ^^^^^^^^^
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


  [?] 工商银行 分红 重试 2/3，等待 4 秒


Exception ignored in: <function tqdm.__del__ at 0x000002BCC5A8EE80>
Traceback (most recent call last):
  File "D:\vibe-coding\trading-topic\labs\.venv\Lib\site-packages\tqdm\std.py", line 1154, in __del__
    self.close()
  File "D:\vibe-coding\trading-topic\labs\.venv\Lib\site-packages\tqdm\notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
    ^^^^^^^^^
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


  [!] 工商银行 分红 失败: ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
    [!] 工商银行 分红数据获取失败

  → 建设银行(601939) 分红记录...
  [?] 建设银行 分红 重试 1/3，等待 2 秒


...

  [!] 中国银行 分红 失败: ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
    [!] 中国银行 分红数据获取失败

  → 招商银行(600036) 分红记录...
  [?] 招商银行 分红 重试 1/3，等待 2 秒


Exception ignored in: <function tqdm.__del__ at 0x000002BCC5A8EE80>
Traceback (most recent call last):
  File "D:\vibe-coding\trading-topic\labs\.venv\Lib\site-packages\tqdm\std.py", line 1154, in __del__
    self.close()
  File "D:\vibe-coding\trading-topic\labs\.venv\Lib\site-packages\tqdm\notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
    ^^^^^^^^^
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


  [?] 招商银行 分红 重试 2/3，等待 4 秒


Exception ignored in: <function tqdm.__del__ at 0x000002BCC5A8EE80>
Traceback (most recent call last):
  File "D:\vibe-coding\trading-topic\labs\.venv\Lib\site-packages\tqdm\std.py", line 1154, in __del__
    self.close()
  File "D:\vibe-coding\trading-topic\labs\.venv\Lib\site-packages\tqdm\notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
    ^^^^^^^^^
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


  [!] 招商银行 分红 失败: ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
    [!] 招商银行 分红数据获取失败

  → 共获取 0 / 5 只股票分红数据


---
## 补充验证：前复权 vs 不复权收盘价对比

以招商银行为例，拉取最近 6 个月的前复权 & 不复权数据，计算复权因子。

$$复权因子 = \frac{不复权收盘价}{前复权收盘价}$$

- 复权因子 = 1：期间无除权除息
- 复权因子 > 1：发生过除权除息，前复权历史价被下调

In [7]:
print("\n" + "=" * 65)
print("  补充验证：前复权 vs 不复权价格对比（招商银行 600036）")
print("=" * 65)

df_raw = safe_call(
    ak.stock_zh_a_hist,
    "招行 不复权",
    symbol="600036",
    period="daily",
    start_date="20260101",
    end_date=END_DATE,
    adjust="",  # 不复权
    retries=2,
    delay=1,
)

if df_raw is not None and not df_raw.empty and "600036" in hist_data:
    df_adj = hist_data["600036"]
    df_raw["日期"] = pd.to_datetime(df_raw["日期"])
    df_raw = df_raw.rename(columns={"收盘": "收盘(不复权)"})

    merged = df_adj[["日期", "收盘"]].merge(
        df_raw[["日期", "收盘(不复权)"]], on="日期", how="inner"
    )

    if not merged.empty:
        merged["复权因子"] = merged["收盘(不复权)"] / merged["收盘"]
        print(f"\n  对比行数: {len(merged)}")
        print(f"  最新日期: {merged['日期'].iloc[-1].strftime('%Y-%m-%d')}")
        print(f"  前复权收盘: {merged['收盘'].iloc[-1]:.2f}")
        print(f"  不复权收盘: {merged['收盘(不复权)'].iloc[-1]:.2f}")
        print(f"  最新复权因子: {merged['复权因子'].iloc[-1]:.4f}")

        csv_path = os.path.join(DATA_DIR, "招商银行_复权对比.csv")
        merged.to_csv(csv_path, index=False, encoding="utf-8-sig")
        print(f"\n  → 对比数据已保存: labs/data/lab1_akshare/招商银行_复权对比.csv")
else:
    print("  [!] 缺少数据，跳过对比")


  补充验证：前复权 vs 不复权价格对比（招商银行 600036）
  [?] 招行 不复权 重试 1/2，等待 1 秒
  [!] 招行 不复权 失败: ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
  [!] 缺少数据，跳过对比


---
## 输出汇总

In [8]:
print(f"\n{'=' * 65}")
print(f"  实验一（akshare 版）完成！")
print(f"  数据目录: labs/data/lab1_akshare/")
print(f"{'=' * 65}")
print(f"\n  输出文件列表:")
for f in sorted(os.listdir(DATA_DIR)):
    fpath = os.path.join(DATA_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"    labs/data/lab1_akshare/{f}  ({size_kb:.1f} KB)")
print(f"{'=' * 65}")


  实验一（akshare 版）完成！
  数据目录: labs/data/lab1_akshare/

  输出文件列表:
    labs/data/lab1_akshare/中国银行_601988_分红记录.csv  (1.8 KB)
    labs/data/lab1_akshare/中国银行_601988_日线_前复权.csv  (85.1 KB)
    labs/data/lab1_akshare/农业银行_601288_分红记录.csv  (1.8 KB)
    labs/data/lab1_akshare/农业银行_601288_日线_前复权.csv  (85.5 KB)
    labs/data/lab1_akshare/工商银行_601398_分红记录.csv  (1.8 KB)
    labs/data/lab1_akshare/工商银行_601398_日线_前复权.csv  (85.7 KB)
    labs/data/lab1_akshare/建设银行_601939_分红记录.csv  (1.7 KB)
    labs/data/lab1_akshare/建设银行_601939_日线_前复权.csv  (84.6 KB)
    labs/data/lab1_akshare/招商银行_600036_分红记录.csv  (1.5 KB)
    labs/data/lab1_akshare/招商银行_600036_日线_前复权.csv  (89.5 KB)
    labs/data/lab1_akshare/招商银行_复权对比.csv  (5.3 KB)
    labs/data/lab1_akshare/银行板块清单.csv  (4.2 KB)
